# Silver Exploration Notebook


## Imports

In [3]:
import os
import findspark

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    lit,
    when,
    to_timestamp,
    from_utc_timestamp,
    dayofweek,
    hour,
    month,
    input_file_name,
    regexp_extract,
    arrays_zip,
    explode,
    size,
    count,
    length,
    min,
    max,
    slice,
    regexp_extract,
    desc,
    unix_micros,
    abs,
    sum as spark_sum
)

## Start Spark

In [4]:
findspark.init()

spark = (
    SparkSession.builder
    .appName("silver-exploration")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/02 11:40:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
RAW = os.path.expanduser("~/spark-lab-data/dev/project/raw")
print(RAW)

/home/ubuntu/spark-lab-data/dev/project/raw


## Read datasets

In [6]:
# Bike-sharing data
trips = (
    spark.read
    .option("header", True)
    .csv(f"{RAW}/trips/*/*.csv")
)

# Weather data
weather = (
    spark.read
    .option("multiLine", True)
    .json(f"{RAW}/weather/*/*.json")
)

## Exporation of Bike-sharing dataset

In [7]:
trips.printSchema()
trips.show(5, truncate=False)
print("Total rows:", trips.count())
print("Columns:", trips.columns)

root
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_description: string (nullable = true)
 |-- start_station_latitude: string (nullable = true)
 |-- start_station_longitude: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_description: string (nullable = true)
 |-- end_station_latitude: string (nullable = true)
 |-- end_station_longitude: string (nullable = true)

+--------------------------------+--------------------------------+--------+----------------+-------------------------+----------------------------+----------------------+-----------------------+--------------+------------------+-----------------------+--------------------+---------------------+
|started_at                      |ended_at               

Total rows: 8432882
Columns: ['started_at', 'ended_at', 'duration', 'start_station_id', 'start_station_name', 'start_station_description', 'start_station_latitude', 'start_station_longitude', 'end_station_id', 'end_station_name', 'end_station_description', 'end_station_latitude', 'end_station_longitude']


- We have a total of 8 428 152 biketrips in Oslo, Trondheim and Bergen.
- The columns in the Bike-sharing dataset include: `started_at`, `endend_at`, `duration`, `start_station_id`, `start_station_name`, `start_station_description`, `start_station_latitude`, `start_station_longitude`, `end_station_id`, `end_station_name`, `end_station_description`, `end_station_latitude`, `end_station_longitude`. 

#### Data Types
The 13 columns are stored as string data types. This is because PySpark reads CSV columns as strings by default. However, Several columns represent numeric values or timestamps. We will therefore convert these columns to appropriate data types. 

- Timestamps: `started_at`and `ended_at` -> timestamp
- Trip duration: `duration` -> integer(maybe double?)
- Coorinates: `latitude`and `longitude`-> double

Maybe:
- StationIDs: `start_station_id` and `stop_station_id` -> keep as string og convert to integer

From the table above we can see the ID is stored as a numeric value, but we would argue to keep them as a string as they are identifiers rather than numeric measurements. This also provides a greater flexibility if future datasets contains letters(e.g. ST123), without requiring changes to the data type. 


### Investigating missing values

In [8]:
trips.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in trips.columns
]).show()

+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+
|started_at|ended_at|duration|start_station_id|start_station_name|start_station_description|start_station_latitude|start_station_longitude|end_station_id|end_station_name|end_station_description|end_station_latitude|end_station_longitude|
+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+
|         0|       0|       0|               0|                 0|                    11018|                     0|                      0|             0|               0|                   6962|                   0|                    0|
+----------+--------+--------+--------------

#### Missing values
The `start_station_description` column contains 11 000 missing values, while `end_station_description` contains 6950 missing values.

#### Is this detrimental to our project or analysis?
These columns are not truly important for the type of analysing we want to do. Our main objective is to investigate how weather conditions influence bike-sharing usage. Since station ID and names are available, the missing descriptions do not prevent us from identifying stations and analyzing trips. 

Although they are not required for our current objectives, they could be valuable for other purposes later. In the Gold layer, we can simply exclude columns that are not relevant to are final analytical dataset. 

### Investigating duplicates

In [9]:
print("Total rows:", trips.count())
print("Duplicate rows:", trips.count() - trips.distinct().count())

Total rows: 8432882


26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:17 WARN RowBasedKeyValueBatch: Calling spill() on

Duplicate rows: 25


Out of the 8 428 152 trips, there may be 25 duplicates. 

Are we sure these are duplicates?

These records contain identical values, but we cannot confirm that they represent duplicate trips. The dataset does not include a unique identifier.

We should therefore investigate these records before deciding to remove them.

In [10]:
duplicates = (
    trips
    .groupBy(trips.columns)
    .count()
    .filter(col("count") > 1)
)

duplicates.show(truncate=False)

26/11/02 12:44:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:44:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:44:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:44:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:44:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:44:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:44:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:44:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:44:09 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------------------+--------------------------------+--------+----------------+---------------------------+-----------------------------------------------------------+----------------------+-----------------------+--------------+---------------------+------------------------------------------------------------------+--------------------+---------------------+-----+
|started_at                      |ended_at                        |duration|start_station_id|start_station_name         |start_station_description                                  |start_station_latitude|start_station_longitude|end_station_id|end_station_name     |end_station_description                                           |end_station_latitude|end_station_longitude|count|
+--------------------------------+--------------------------------+--------+----------------+---------------------------+-----------------------------------------------------------+----------------------+-----------------------+--------

In [11]:
duplicates.select(
    "started_at",
    "ended_at",
    "duration",
    "start_station_name",
    "end_station_name",
    "start_station_id",
    "count"
).orderBy(col("count").desc()).show(50, truncate=False)

26/11/02 12:48:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:48:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:48:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:48:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:48:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:48:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:48:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:48:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 12:48:25 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------------------+--------------------------------+--------+---------------------------+---------------------+----------------+-----+
|started_at                      |ended_at                        |duration|start_station_name         |end_station_name     |start_station_id|count|
+--------------------------------+--------------------------------+--------+---------------------------+---------------------+----------------+-----+
|2022-11-19 20:52:38.612000+00:00|2022-11-19 20:56:22.030000+00:00|223     |Østre Murallmenningen nedre|C. Sundts gate       |816             |2    |
|2022-09-14 15:11:04.177000+00:00|2022-09-14 15:19:44.846000+00:00|520     |Møllendalsplass            |Busstasjonen 3  Øst  |220             |2    |
|2024-09-17 17:01:36.002000+00:00|2024-09-17 17:09:04.118000+00:00|448     |The Hub                    |Olaf Ryes plass      |2328            |2    |
|2024-04-29 11:22:12.751000+00:00|2024-04-29 11:28:06.534000+00:00|353     |Torvet I                

### Investigatin values and distributions

#### First Duration

In [32]:
t = trips.withColumn("duration_s", col("duration").cast("double"))

t.select("duration_s").summary("min", "25%", "50%", "75%", "max").show()

t.select(spark_sum(when(col("duration_s") <= 0, 1).otherwise(0)).alias("non_positive"),
         spark_sum(when(col("duration_s") < 60, 1).otherwise(0)).alias("under_60s"),
         spark_sum(when(col("duration_s").isNull(), 1).otherwise(0)).alias("null_duration")).show()

+-------+----------+
|summary|duration_s|
+-------+----------+
|    min|       0.0|
|    25%|     341.0|
|    50%|     524.0|
|    75%|     806.0|
|    max|   79790.0|
+-------+----------+



+------------+---------+-------------+
|non_positive|under_60s|null_duration|
+------------+---------+-------------+
|           2|     1273|            0|
+------------+---------+-------------+



#### Durations to investigate ferther
- Is it possible that a trip last around 22 hours?
- 2 trips has a negative durations

In [34]:
print("Number of trips longer than 12 hours: ", t.filter(col("duration_s") > 12 * 3600).count())

t.select(
    "started_at",
    "ended_at",
    "duration_s"
).orderBy(desc("duration_s")).show(15, truncate=False)

Number of trips longer than 12 hours:  15


+-----------------------+-----------------------+----------+
|started_at             |ended_at               |duration_s|
+-----------------------+-----------------------+----------+
|2026-06-19 13:38:13.233|2026-06-20 11:48:03.076|79790.0   |
|2025-10-17 14:35:20.232|2025-10-18 11:15:52.091|74431.0   |
|2022-04-16 12:47:44.363|2022-04-17 07:37:10.178|67765.0   |
|2023-04-07 13:40:11.612|2023-04-08 08:18:37.147|67105.0   |
|2023-05-31 15:31:03.704|2023-06-01 08:37:01.096|61557.0   |
|2026-05-04 18:28:16.356|2026-05-05 11:14:23.84 |60367.0   |
|2022-11-01 15:19:05.897|2022-11-02 07:26:44.435|58058.0   |
|2022-11-24 14:20:12.725|2022-11-25 05:44:40.784|55468.0   |
|2026-08-06 20:38:28.625|2026-08-07 11:37:01.191|53913.0   |
|2022-05-20 16:49:31.692|2022-05-21 06:44:45.835|50114.0   |
|2022-10-10 19:37:20.877|2022-10-11 09:10:35.12 |48794.0   |
|2022-05-15 15:36:45.755|2022-05-16 04:41:07.477|47061.0   |
|2026-10-19 16:33:30.177|2026-10-20 05:36:35.162|46985.0   |
|2022-11-15 15:53:39.71 

How long those the battery on the city-bike last?

Whats the most plausibel reason for this? 
- Maybe they forgot to "stamp out" the bike?
- 

#### Removing trips that has a negative duration

In [35]:
trips_silver = t.filter(col("duration_s") > 0)

#### Does duration correspond to the start and end time?

In [58]:
trips.select(
    "started_at",
    "ended_at",
    "duration",
    # does started_at and ended_at match the duration?
    (to_timestamp(col("ended_at")).cast("long") - to_timestamp(col("started_at")).cast("long")).alias("duration_check")
    ).show(5, truncate=False)

+-----------------------+-----------------------+--------+--------------+
|started_at             |ended_at               |duration|duration_check|
+-----------------------+-----------------------+--------+--------------+
|2022-08-01 03:08:45.036|2022-08-01 03:11:38.571|173     |173           |
|2022-08-01 03:09:05.419|2022-08-01 03:18:15.885|550     |550           |
|2022-08-01 03:22:25.178|2022-08-01 03:30:57.051|511     |512           |
|2022-08-01 03:29:20.355|2022-08-01 03:38:43.156|562     |563           |
|2022-08-01 03:29:25.883|2022-08-01 03:37:00.311|454     |455           |
+-----------------------+-----------------------+--------+--------------+
only showing top 5 rows



Since we used the `.cast("long")`we lose miliseconds, which can explain the reason for the 1 second difference. 

for example:
- 511 and 512
- 562 and 563
- 454 and 455

Let's investigate the duration a bit further.

In [62]:
trips_check = trips.withColumn(
    "duration_diff",
    abs(
        (unix_micros(to_timestamp("ended_at"))
         - unix_micros(to_timestamp("started_at"))) / 1_000_000
        - col("duration").cast("double")
    )
)

trips_check.select("duration_diff").summary().show()

+-------+-------------------+
|summary|      duration_diff|
+-------+-------------------+
|  count|            8428152|
|   mean|0.45660269961908617|
| stddev| 0.2853735155457395|
|    min|                0.0|
|    25%|0.21299999999996544|
|    50%|0.42699999999990723|
|    75%| 0.6979999999999791|
|    max| 0.9990000000000236|
+-------+-------------------+



Since the maximum difference are below one second, we consider the recorded durations consistent with the timestamps. 

### Investigating the timestamp format

In [2]:
trips.groupBy(length("started_at").alias("len")).count().orderBy("len").show()
trips.select("started_at", "ended_at", "duration").distinct().limit(10).show(truncate=False)

NameError: name 'trips' is not defined

In [44]:
# Setting the spark session timezone to UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Converting the started_at and ended_at columns to timestamp type
trips = (
    trips
    .withColumn("started_ts", to_timestamp(col("started_at")))
    .withColumn("ended_ts", to_timestamp(col("ended_at")))
)

# Did the conversion work?
trips.filter(
    col("started_ts").isNull() |
    col("ended_ts").isNull()
).count()

# Let's compare the converted timestamps
trips.select(
    "started_at",
    "started_ts",
    "ended_at",
    "ended_ts"
).show(10, truncate=False)

+-----------------------+-----------------------+-----------------------+-----------------------+
|started_at             |started_ts             |ended_at               |ended_ts               |
+-----------------------+-----------------------+-----------------------+-----------------------+
|2022-08-01 03:08:45.036|2022-08-01 03:08:45.036|2022-08-01 03:11:38.571|2022-08-01 03:11:38.571|
|2022-08-01 03:09:05.419|2022-08-01 03:09:05.419|2022-08-01 03:18:15.885|2022-08-01 03:18:15.885|
|2022-08-01 03:22:25.178|2022-08-01 03:22:25.178|2022-08-01 03:30:57.051|2022-08-01 03:30:57.051|
|2022-08-01 03:29:20.355|2022-08-01 03:29:20.355|2022-08-01 03:38:43.156|2022-08-01 03:38:43.156|
|2022-08-01 03:29:25.883|2022-08-01 03:29:25.883|2022-08-01 03:37:00.311|2022-08-01 03:37:00.311|
|2022-08-01 03:30:15.563|2022-08-01 03:30:15.563|2022-08-01 03:45:27.332|2022-08-01 03:45:27.332|
|2022-08-01 03:36:52.062|2022-08-01 03:36:52.062|2022-08-01 03:50:05.634|2022-08-01 03:50:05.634|
|2022-08-01 03:38:34

### Converting Timestamps to Norwegian Local Time

The `started_ts` and `ended_ts` timestamps are recorded in UTC, as indicated by `+00:00`.

For example:
- `started_at`: 2022-08-01 05:14:13.781000+00:00
- `ended_at`: 2022-08-01 05:26:27.585000+00:00

We convert these timestamps to Norwegian local time (`Europe/Oslo`), accounting for daylight saving time. The original UTC timestamps are preserved for further processing.

In [45]:
trips = (
    trips
    .withColumn(
        "started_local",
        from_utc_timestamp(
            to_timestamp(col("started_ts")),
            "Europe/Oslo"
        )
    )
    .withColumn(
        "ended_local",
        from_utc_timestamp(
            to_timestamp(col("ended_ts")),
            "Europe/Oslo"
        )
    )
)

trips.select("started_ts","started_local","ended_ts","ended_local").show(10, truncate=False)

+-----------------------+-----------------------+-----------------------+-----------------------+
|started_ts             |started_local          |ended_ts               |ended_local            |
+-----------------------+-----------------------+-----------------------+-----------------------+
|2022-08-01 03:08:45.036|2022-08-01 05:08:45.036|2022-08-01 03:11:38.571|2022-08-01 05:11:38.571|
|2022-08-01 03:09:05.419|2022-08-01 05:09:05.419|2022-08-01 03:18:15.885|2022-08-01 05:18:15.885|
|2022-08-01 03:22:25.178|2022-08-01 05:22:25.178|2022-08-01 03:30:57.051|2022-08-01 05:30:57.051|
|2022-08-01 03:29:20.355|2022-08-01 05:29:20.355|2022-08-01 03:38:43.156|2022-08-01 05:38:43.156|
|2022-08-01 03:29:25.883|2022-08-01 05:29:25.883|2022-08-01 03:37:00.311|2022-08-01 05:37:00.311|
|2022-08-01 03:30:15.563|2022-08-01 05:30:15.563|2022-08-01 03:45:27.332|2022-08-01 05:45:27.332|
|2022-08-01 03:36:52.062|2022-08-01 05:36:52.062|2022-08-01 03:50:05.634|2022-08-01 05:50:05.634|
|2022-08-01 03:38:34

### Number of trips divided by the different cities

In [15]:
trips = (trips
         .withColumn("src", input_file_name())
         .withColumn("city", regexp_extract("src", r"/trips/([^/]+)/", 1))
         .withColumn("year_month", regexp_extract("src", r"/([0-9]{4}-[0-9]{2})\.csv", 1)))

trips.groupBy("city").count().show()

+---------+-------+
|     city|  count|
+---------+-------+
|     oslo|5718138|
|   bergen|1904236|
|trondheim| 805778|
+---------+-------+



In [16]:
trips.filter(length("started_at") == 25).select("started_at", "ended_at").show(5, truncate=False)

p = (trips
     .withColumn("start_ts", to_timestamp("started_at"))
     .withColumn("end_ts", to_timestamp("ended_at"))
     .withColumn("len_start", length("started_at"))
     .withColumn("len_end", length("ended_at")))

p.groupBy("len_start", "len_end").count().show()
p.filter(col("start_ts").isNull() | col("end_ts").isNull()).count()

from pyspark.sql.functions import regexp_extract, input_file_name, length, count, sum as spark_sum, when

chk = (trips
       .withColumn("src", input_file_name())
       .withColumn("city", regexp_extract("src", r"/trips/([^/]+)/", 1))
       .withColumn("year_month", regexp_extract("src", r"/([0-9]{4}-[0-9]{2})\.csv", 1))
       .groupBy("city", "year_month")
       .agg(count("*").alias("rows"),
            spark_sum(when(length("started_at") == 25, 1).otherwise(0)).alias("short_start")))

chk.withColumn("pct", col("short_start") / col("rows") * 100) \
   .orderBy(col("pct").desc()).show(10)

+----------+--------+
|started_at|ended_at|
+----------+--------+
+----------+--------+



+---------+-------+-------+
|len_start|len_end|  count|
+---------+-------+-------+
|       22|     23| 683713|
|       23|     21|  68263|
|       22|     19|    700|
|       23|     23|6824440|
|       19|     22|    801|
|       22|     22|  68341|
|       23|     19|   7296|
|       21|     23|  68614|
|       19|     23|   7321|
|       21|     19|     64|
|       21|     22|   6794|
|       21|     21|    689|
|       23|     22| 684235|
|       22|     21|   6810|
|       19|     21|     66|
|       19|     19|      5|
+---------+-------+-------+



+----+----------+------+-----------+---+
|city|year_month|  rows|short_start|pct|
+----+----------+------+-----------+---+
|oslo|   2022-06|195454|          0|0.0|
|oslo|   2022-08|207333|          0|0.0|
|oslo|   2023-06|192584|          0|0.0|
|oslo|   2024-08|177594|          0|0.0|
|oslo|   2025-08|176415|          0|0.0|
|oslo|   2026-08|187373|          0|0.0|
|oslo|   2023-08|172110|          0|0.0|
|oslo|   2024-06|168176|          0|0.0|
|oslo|   2024-05|170068|          0|0.0|
|oslo|   2022-09|165877|          0|0.0|
+----+----------+------+-----------+---+
only showing top 10 rows



In [17]:
t = trips.withColumn("duration_s", col("duration").cast("double"))
print(t.approxQuantile("duration_s", [0.5, 0.99, 0.999, 0.9999], 0.001))
print(t.filter(col("duration_s") > 6*3600).count())
print(t.filter(col("duration_s") > 12*3600).count())
print(t.approxQuantile("duration_s", [0.999, 0.9999], 0.0001))

[524.0, 3706.0, 79790.0, 79790.0]


315


15


[9142.0, 79790.0]


In [18]:
total = trips.count()
distinct = trips.dropDuplicates().count()
print(total, distinct, total - distinct)

26/11/02 07:58:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 07:58:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 07:58:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 07:58:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 07:58:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 07:58:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 07:58:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 07:58:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 07:58:34 WARN RowBasedKeyValueBatch: Calling spill() on

8428152 8428127 25


## Exporation of Weather dataset

In [19]:
weather.printSchema()
print("Weather files:", weather.count())

root
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- hourly: struct (nullable = true)
 |    |-- apparent_temperature: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- cloud_cover: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- precipitation: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- rain: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- relative_humidity_2m: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- snowfall: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- weather_code: array (nullable = true)
 |    |    |-- el

In [20]:
from pyspark.sql.functions import arrays_zip, explode

w = (weather
     .withColumn("src", input_file_name())
     .select("src", explode(arrays_zip(
         col("hourly.time"),
         col("hourly.temperature_2m"),
         col("hourly.rain"),
         col("hourly.wind_speed_10m"),
        col("hourly.apparent_temperature"),
        col("hourly.precipitation")
     )).alias("h")))

w.select("src", "h.*").show(3, truncate = False)

+-----------------------------------------------------------------------------------------------------------------------+----------------+--------------+----+--------------+--------------------+-------------+
|src                                                                                                                    |time            |temperature_2m|rain|wind_speed_10m|apparent_temperature|precipitation|
+-----------------------------------------------------------------------------------------------------------------------+----------------+--------------+----+--------------+--------------------+-------------+
|file:///home/ubuntu/spark-lab-data/dev/project/raw/weather/trondheim/2022-01-01_2026-10-18_20261019T112729Z-52a0eb.json|2022-01-01T00:00|0.5           |0.0 |7.7           |-3.0                |0.0          |
|file:///home/ubuntu/spark-lab-data/dev/project/raw/weather/trondheim/2022-01-01_2026-10-18_20261019T112729Z-52a0eb.json|2022-01-01T01:00|0.2           |0.0 |7.6   

In [21]:
weather.select(
    "timezone",
    slice("hourly.time", 1, 5).alias("first_timestamps")
).show(10, truncate=False)

+--------+------------------------------------------------------------------------------------------+
|timezone|first_timestamps                                                                          |
+--------+------------------------------------------------------------------------------------------+
|GMT     |[2022-01-01T00:00, 2022-01-01T01:00, 2022-01-01T02:00, 2022-01-01T03:00, 2022-01-01T04:00]|
|GMT     |[2022-01-01T00:00, 2022-01-01T01:00, 2022-01-01T02:00, 2022-01-01T03:00, 2022-01-01T04:00]|
|GMT     |[2022-01-01T00:00, 2022-01-01T01:00, 2022-01-01T02:00, 2022-01-01T03:00, 2022-01-01T04:00]|
|GMT     |[2026-10-09T00:00, 2026-10-09T01:00, 2026-10-09T02:00, 2026-10-09T03:00, 2026-10-09T04:00]|
|GMT     |[2026-10-09T00:00, 2026-10-09T01:00, 2026-10-09T02:00, 2026-10-09T03:00, 2026-10-09T04:00]|
|GMT     |[2026-10-06T00:00, 2026-10-06T01:00, 2026-10-06T02:00, 2026-10-06T03:00, 2026-10-06T04:00]|
|GMT     |[2026-10-09T00:00, 2026-10-09T01:00, 2026-10-09T02:00, 2026-10-09T03:00,

In [22]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

weather_silver = (
    weather
    .withColumn(
        "hourly_data",
        explode(arrays_zip(
            "hourly.time",
            "hourly.temperature_2m",
            "hourly.precipitation",
            "hourly.wind_speed_10m"
        ))
    )
    .select(
        col("latitude"),
        col("longitude"),
        col("hourly_data.time").alias("time"),
        col("hourly_data.temperature_2m").alias("temperature_2m"),
        col("hourly_data.precipitation").alias("precipitation"),
        col("hourly_data.wind_speed_10m").alias("wind_speed_10m")
    )
    .withColumn("time_utc", to_timestamp("time"))
    .withColumn(
        "time_local",
        from_utc_timestamp("time_utc", "Europe/Oslo")
    )
    .drop("time")
)

weather_silver.show(10, truncate=False)
weather_silver.printSchema()

+---------+---------+--------------+-------------+--------------+-------------------+-------------------+
|latitude |longitude|temperature_2m|precipitation|wind_speed_10m|time_utc           |time_local         |
+---------+---------+--------------+-------------+--------------+-------------------+-------------------+
|63.444637|10.366492|0.5           |0.0          |7.7           |2022-01-01 00:00:00|2022-01-01 01:00:00|
|63.444637|10.366492|0.2           |0.0          |7.6           |2022-01-01 01:00:00|2022-01-01 02:00:00|
|63.444637|10.366492|-0.2          |0.0          |5.9           |2022-01-01 02:00:00|2022-01-01 03:00:00|
|63.444637|10.366492|-0.7          |0.0          |5.5           |2022-01-01 03:00:00|2022-01-01 04:00:00|
|63.444637|10.366492|-3.6          |0.0          |6.9           |2022-01-01 04:00:00|2022-01-01 05:00:00|
|63.444637|10.366492|-1.2          |0.0          |5.4           |2022-01-01 05:00:00|2022-01-01 06:00:00|
|63.444637|10.366492|-0.7          |0.0       

##### Check if any weather values are missing

In [23]:
from pyspark.sql.functions import sum as spark_sum, when

weather_cols = ["temperature_2m", "rain", "wind_speed_10m",
                "apparent_temperature", "precipitation"]

wx.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in weather_cols
]).show()

NameError: name 'wx' is not defined